# Atividade - Perceptron e KNN

------------------

## Desafio 1: Classificação Binária com Perceptron Treinável

**Variáveis de entrada:**
- x1: Valor da Transação Normalizado (0.0 a 10.0)
- x2: Frequência de Operações Recentes

**Saída:** 0 = Legítima | 1 = Suspeita

Imports + dados:

In [10]:
import numpy as np

# Matriz de treino: [Valor Normalizado (0.0 a 10.0), Frequência de Operações]
X_treino_fraude = np.array([
    [1.5, 1.0],
    [2.0, 2.0],
    [3.5, 1.5],
    [3.0, 3.0],
    [6.5, 5.0],
    [7.0, 7.0],
    [8.5, 6.0],
    [9.0, 8.0]
])

# Rótulos (0: Legítima, 1: Suspeita)
y_treino_fraude = np.array([0, 0, 0, 0, 1, 1, 1, 1])

print("Amostras de treino:", X_treino_fraude.shape[0])
print("Features por amostra:", X_treino_fraude.shape[1])

Amostras de treino: 8
Features por amostra: 2


Funcoes

In [11]:
def treinar_perceptron(X, y, taxa_aprendizado, epocas):
    """
    Treina o Perceptron pela regra de Rosenblatt.
    Inicializa pesos e bias em 1.0 e ajusta a cada amostra com erro.
    """
    w = np.ones(X.shape[1])
    b = 1.0

    for epoca in range(epocas):
        erros = 0
        for x_i, y_real in zip(X, y):
            y_pred = 1 if (np.dot(x_i, w) + b) >= 0 else 0
            erro = y_real - y_pred

            if erro != 0:
                w += taxa_aprendizado * erro * x_i
                b += taxa_aprendizado * erro
                erros += 1

        if erros == 0:
            print(f"Modelo convergiu na época {epoca + 1}.")
            break

    return w, b


def prever_fraude(entradas, pesos, bias):
    """
    Calcula a combinação linear e aplica a função degrau.
    """
    z = np.dot(entradas, pesos) + bias
    return 1 if z >= 0 else 0

Execucao

In [12]:
taxa = 0.1
epocas = 100

pesos_fraude, bias_fraude = treinar_perceptron(
    X_treino_fraude, y_treino_fraude, taxa, epocas
)

print(f"Pesos finais: w1 = {pesos_fraude[0]:.2f}, w2 = {pesos_fraude[1]:.2f}")
print(f"Bias final:   b = {bias_fraude:.2f}")

transacao_A = np.array([5.0, 4.0])
transacao_B = np.array([7.0, 6.5])

classe_A = prever_fraude(transacao_A, pesos_fraude, bias_fraude)
classe_B = prever_fraude(transacao_B, pesos_fraude, bias_fraude)

diagnostico = {0: "Transação Legítima", 1: "Transação Suspeita"}

print(f"\nTransação A {transacao_A.tolist()} -> {diagnostico[classe_A]}")
print(f"Transação B {transacao_B.tolist()} -> {diagnostico[classe_B]}")

Modelo convergiu na época 12.
Pesos finais: w1 = -0.00, w2 = 0.40
Bias final:   b = -1.20

Transação A [5.0, 4.0] -> Transação Suspeita
Transação B [7.0, 6.5] -> Transação Suspeita


Obs: O modelo convergiu para uma fronteira vertical em x2 ≈ 3.0, pois as features x1 e x2 são correlacionadas. A Transação A é classificada como suspeita por ter frequência (4.0) acima do limiar aprendido. Isso é coerente com os dados de treino.

## Desafio 2: Predição de Risco de Churn com Classificador KNN

**Variáveis de entrada:**
- x1: Dias de inatividade
- x2: Chamados Críticos em Aberto

**Saída:** 0 = Baixo Risco | 1 = Alto Risco

Imports + dados

In [13]:
import numpy as np

# Matriz de treino: [Dias de Inatividade, Chamados Críticos]
X_treino_churn = np.array([
    [2.0, 1.0],
    [3.0, 0.0],
    [5.0, 1.0],
    [6.0, 2.0],
    [15.0, 4.0],
    [18.0, 5.0],
    [20.0, 4.0],
    [22.0, 6.0]
])

# Rótulos (0: Baixo Risco, 1: Alto Risco)
y_treino_churn = np.array([0, 0, 0, 0, 1, 1, 1, 1])

Funcoes

In [14]:
def calcular_distancias(X, ponto, metrica):
    """
    Calcula as distâncias do ponto de consulta para todas as amostras de X.
    Suporta as métricas euclidiana (ord=2) e manhattan (ord=1).
    """
    if metrica == "euclidiana":
        return np.linalg.norm(X - ponto, ord=2, axis=1)
    elif metrica == "manhattan":
        return np.linalg.norm(X - ponto, ord=1, axis=1)
    else:
        raise ValueError("Métrica inválida. Use 'euclidiana' ou 'manhattan'.")


def knn_classificar(X, y, ponto, k, metrica):
    """
    Classifica um ponto por votação dos k vizinhos mais próximos.
    Retorna a classe predita, os índices dos vizinhos e as distâncias.
    """
    distancias = calcular_distancias(X, ponto, metrica)
    vizinhos = np.argsort(distancias)[:k]
    votos = y[vizinhos]
    classe_predita = int(np.bincount(votos).argmax())

    return classe_predita, vizinhos, distancias[vizinhos]

Execucao

In [15]:
clientes = {
    "Cliente 1": np.array([13.0, 10.0]),
    "Cliente 2": np.array([7.0, 1.5])
}

k = 3
diagnostico = {0: "Baixo Risco", 1: "Alto Risco"}

for metrica in ["euclidiana", "manhattan"]:
    print(f"===== Distância {metrica.capitalize()} (k = {k}) =====")
    for nome, ponto in clientes.items():
        classe, vizinhos, dists = knn_classificar(
            X_treino_churn, y_treino_churn, ponto, k, metrica
        )
        print(f"{nome} {ponto.tolist()} -> {diagnostico[classe]}")
        print(f"  Índices dos vizinhos: {vizinhos.tolist()}")
        print(f"  Distâncias: {np.round(dists, 4).tolist()}")
    print()

===== Distância Euclidiana (k = 3) =====
Cliente 1 [13.0, 10.0] -> Alto Risco
  Índices dos vizinhos: [4, 5, 6]
  Distâncias: [6.3246, 7.0711, 9.2195]
Cliente 2 [7.0, 1.5] -> Baixo Risco
  Índices dos vizinhos: [3, 2, 1]
  Distâncias: [1.118, 2.0616, 4.272]

===== Distância Manhattan (k = 3) =====
Cliente 1 [13.0, 10.0] -> Alto Risco
  Índices dos vizinhos: [4, 5, 7]
  Distâncias: [8.0, 10.0, 13.0]
Cliente 2 [7.0, 1.5] -> Baixo Risco
  Índices dos vizinhos: [3, 2, 1]
  Distâncias: [1.5, 2.5, 5.5]



# Desafio 3 — Recomendação de Servidores Cloud por Similaridade Espacial

**Variáveis de entrada (por instância):**
- x1 — vCPUs
- x2 — Memória RAM (GB)
- x3 — Armazenamento SSD NVMe (GB)

**Abordagem:** KNN como recomendador — sem votação, apenas ordenação por proximidade geométrica.

Catalogo + nomes

In [16]:
import numpy as np

# Catálogo: [vCPUs, Memória RAM (GB), Armazenamento SSD (GB)]
catalogo_servidores = np.array([
    [2.0, 4.0, 50.0],
    [4.0, 8.0, 100.0],
    [8.0, 16.0, 250.0],
    [16.0, 32.0, 500.0],
    [32.0, 64.0, 1000.0],
    [64.0, 128.0, 2000.0]
])

nomes_servidores = [
    "Micro Instância Web",
    "Standard App Server",
    "Medium Backend & Cache",
    "Database Enterprise",
    "High Performance Computing",
    "Big Data & AI Training"
]

Funcao de recomendacao

In [17]:
def recomendar_servidores(catalogo, perfil, k, metrica):
    """
    Recomenda as k instâncias mais similares ao perfil informado.
    Retorna os índices das instâncias e as distâncias calculadas.
    """
    if metrica == "euclidiana":
        distancias = np.linalg.norm(catalogo - perfil, ord=2, axis=1)
    elif metrica == "manhattan":
        distancias = np.linalg.norm(catalogo - perfil, ord=1, axis=1)
    else:
        raise ValueError("Métrica inválida. Use 'euclidiana' ou 'manhattan'.")

    indices = np.argsort(distancias)[:k]

    return indices, distancias[indices]

Execucao e ranking

In [18]:
perfil_demanda = np.array([12.0, 28.0, 850.0])
k = 2

indices, distancias = recomendar_servidores(
    catalogo_servidores, perfil_demanda, k, "euclidiana"
)

print(f"Perfil demandado: {perfil_demanda.tolist()}")
print(f"Top {k} servidores mais similares:\n")

for pos, (idx, dist) in enumerate(zip(indices, distancias), start=1):
    specs = catalogo_servidores[idx]
    print(f"{pos}º lugar: {nomes_servidores[idx]}")
    print(f"  Especificacoes: {specs[0]:.0f} vCPUs | {specs[1]:.0f} GB RAM | {specs[2]:.0f} GB SSD")
    print(f"  Distância euclidiana: {dist:.4f}")
    print()

Perfil demandado: [12.0, 28.0, 850.0]
Top 2 servidores mais similares:

1º lugar: High Performance Computing
  Especificacoes: 32 vCPUs | 64 GB RAM | 1000 GB SSD
  Distância euclidiana: 155.5506

2º lugar: Database Enterprise
  Especificacoes: 16 vCPUs | 32 GB RAM | 500 GB SSD
  Distância euclidiana: 350.0457

